# Taller · Destilación en pequeño
**Un LLM grande (profesor, API de OpenAI) enseña a un modelo de 135 M parámetros (estudiante) que corre en tu CPU.**

Máster en IA · UAI · Tópicos Avanzados en IA 2

Taller de **15 horas en clases, entrega individual** (ver `Guia_Taller_Destilacion.pdf`). Este notebook recorre el ciclo técnico: una corrida completa toma **8–15 minutos** en un notebook sin GPU, y lo repetirás con tu propio dominio y distintos ajustes a lo largo de los bloques.

| Etapa | Qué pasa |
|---|---|
| 1 | Configurar: leer `.env`, elegir profesor (OpenAI u offline) y dispositivo |
| 2 | El profesor escribe preguntas y respuestas (entrenamiento + examen) |
| 3 | Cargar el estudiante pequeño y agregarle adaptadores LoRA |
| 4 | Examen **antes** de entrenar: el juez califica de 1 a 10 |
| 5 | Entrenar (SFT con LoRA) sobre las respuestas del profesor |
| 6 | Examen **después** + gráfico + ejemplos lado a lado |

> Sin clave de OpenAI o sin internet: en la celda 1 cambia `SIN_INTERNET = True`. Todo corre igual con un profesor simulado y un modelo diminuto local.

## 1 · Configurar

In [ ]:
SIN_INTERNET = False        # True = sin clave ni descargas (profesor simulado + modelo diminuto)
ESTUDIANTE   = None         # None = HuggingFaceTB/SmolLM2-135M-Instruct · otra opción: "HuggingFaceTB/SmolLM2-360M-Instruct"

import destila as D
cfg = D.configurar(sin_internet=SIN_INTERNET, estudiante=ESTUDIANTE)
profesor = D.ProfesorOffline(cfg) if cfg["profesor_modo"] == "offline" else D.Profesor(cfg)
cfg

**Para discutir:** ¿qué parámetros de `cfg` cambiarías para tu propio caso de negocio? El dominio y los temas son lo primero (`D.CONFIG` en `destila.py`).

## 2 · El profesor genera los datos
El profesor escribe preguntas por tema y luego las responde. Una parte se reserva como **examen fijo**: nunca se entrena con ella, para que la nota antes/después sea comparable.

In [ ]:
entrenamiento, examen = D.generar_datos(cfg, profesor)
import pandas as pd
pd.DataFrame(entrenamiento).head(5)

## 3 · Cargar el estudiante
SmolLM2-135M-Instruct: arquitectura Llama, 135 M parámetros, 270 MB. LoRA agrega matrices pequeñas entrenables; el modelo base queda congelado.

In [ ]:
estudiante = D.cargar_estudiante(cfg)

## 4 · Examen ANTES de entrenar
El estudiante responde el examen y el juez (el mismo profesor, con una rúbrica) califica cada respuesta de 1 a 10 comparándola con la referencia.

In [ ]:
antes = D.evaluar(cfg, estudiante, profesor, examen, "antes")
pd.DataFrame(antes["detalle"])[["tema", "pregunta", "respuesta_estudiante", "nota", "comentario"]].head(6)

## 5 · Entrenar
Ajuste fino supervisado: el modelo aprende a producir la **respuesta** del profesor (la pregunta se enmascara con −100 y no cuenta en la pérdida).

In [ ]:
historial = D.entrenar(cfg, estudiante, entrenamiento)

## 6 · Examen DESPUÉS y comparación

In [ ]:
despues = D.evaluar(cfg, estudiante, profesor, examen, "despues")
ruta = D.comparar(antes, despues, historial, profesor)
from IPython.display import Image, Markdown, display
display(Image(filename=str(ruta)))
display(Markdown((D.SALIDAS / "resultados.md").read_text(encoding="utf-8")))

## 7 · Conversa con tu modelo destilado
Compara con el modelo base (sin entrenar). Prueba también una pregunta **fuera** del dominio: ¿qué pasa?

In [ ]:
D.chatear(estudiante, "Explica qué es LoRA y para qué sirve en un banco.")

In [ ]:
D.chatear(estudiante, "¿Qué es una alucinación y cómo la controlo con RAG en una clínica?")

## Plan de los cinco bloques y entrega
| Bloque | Objetivo | Hito |
|---|---|---|
| 1 | Instalar y correr el ciclo completo | `resultados.png` de la configuración por defecto |
| 2 | Tu dominio y tus datos (`D.CONFIG`, revisar 10 respuestas del profesor) | Secciones 1–2 del informe |
| 3 | Línea base y ≥ 3 experimentos (una variable por vez) | Tabla de experimentos |
| 4 | Auditar al juez (10 ítems a mano) y fijar la corrida final | Sección 5 del informe, `salidas/checkpoint` |
| 5 | Demostración de 5 min y reflexión | Informe PDF + carpeta del proyecto |

**Entrega individual:** informe PDF según `PLANTILLA_INFORME.md` (6–10 páginas) + carpeta del proyecto comprimida (sin `.env`).
Atraso: −1,5 puntos por día hasta nota 1,0. La rúbrica está en la guía.

> Consejo: cada vez que cambies el dominio o la cantidad de datos, borra la carpeta `datos/` para que el profesor los regenere. Guarda cada `salidas/` con un nombre distinto (`salidas_corrida1`, …) para no perder tus experimentos.